In [8]:
import numpy as np
import pandas as pd
import itertools
from scipy.optimize import root
import sys

VOC = 38     
ISC = 8      
Ns = 54      

def extract_villalva_parameters(Voc_n, Isc_n, Ns, n=1.3):
    q, k, T_n = 1.60e-19, 1.38e-23, 298.15
    Vt = (Ns * k * T_n) / q
    Rp = (Voc_n / 0.05) / Isc_n
    Rs = 0.01 * (Voc_n / Isc_n)
    I0_n = (Isc_n - (Voc_n - Isc_n * Rs) / Rp) / (np.exp(Voc_n / (n * Vt)) - 1.0)
    return [Voc_n, Isc_n, Rs, Rp, I0_n, Vt]

pv_params = extract_villalva_parameters(VOC, ISC, Ns)

def solve_module_current_with_bypass(V, G, pv_params, n=1.3):
    Voc_n, Isc_n, Rs, Rp, I0_n, Vt = pv_params
    Ipv = (G / 1000.0) * Isc_n if G > 1.0 else 0.0
    
    I_guess = Ipv
    for _ in range(6): # Fast convergence cap
        vd_over_avt = (V + I_guess * Rs) / (n * Vt)
        vd_over_avt = min(50.0, max(-50.0, vd_over_avt)) 
        expr = np.exp(vd_over_avt)
        f = Ipv - I0_n * (expr - 1.0) - (V + I_guess * Rs) / Rp - I_guess
        df = -I0_n * (Rs / (n * Vt)) * expr - (Rs / Rp) - 1.0
        I_guess -= f / df
        
    I0_bd = 1e-6
    Vt_bd = 0.026 
    v_bd_ratio = -V / Vt_bd
    v_bd_ratio = min(50.0, max(-50.0, v_bd_ratio))
    I_bypass = I0_bd * (np.exp(v_bd_ratio) - 1.0)
    return I_guess + I_bypass

def solve_mna_network(V_array, shading_matrix, switches, pv_params, initial_guess):
    flat_G = np.array(shading_matrix).flatten()

    def residual(variables):
        v10, v11, v12 = variables[0], variables[1], variables[2]
        v20, v21, v22 = variables[3], variables[4], variables[5]
        i_s1a, i_s1b  = variables[6], variables[7]
        i_s2a, i_s2b  = variables[8], variables[9]

        I0 = solve_module_current_with_bypass(V_array - v10, flat_G[0], pv_params)
        I1 = solve_module_current_with_bypass(V_array - v11, flat_G[1], pv_params)
        I2 = solve_module_current_with_bypass(V_array - v12, flat_G[2], pv_params)
        I3 = solve_module_current_with_bypass(v10 - v20, flat_G[3], pv_params)
        I4 = solve_module_current_with_bypass(v11 - v21, flat_G[4], pv_params)
        I5 = solve_module_current_with_bypass(v12 - v22, flat_G[5], pv_params)
        I6 = solve_module_current_with_bypass(v20 - 0, flat_G[6], pv_params)
        I7 = solve_module_current_with_bypass(v21 - 0, flat_G[7], pv_params)
        I8 = solve_module_current_with_bypass(v22 - 0, flat_G[8], pv_params)

        res = np.zeros(10)
        res[0] = I0 - I3 - i_s1a             
        res[1] = I1 - I4 + i_s1a - i_s1b     
        res[2] = I2 - I5 + i_s1b             
        res[3] = I3 - I6 - i_s2a             
        res[4] = I4 - I7 + i_s2a - i_s2b     
        res[5] = I5 - I8 + i_s2b             
        res[6] = v10 - v11 if switches['S1_a'] else i_s1a
        res[7] = v11 - v12 if switches['S1_b'] else i_s1b
        res[8] = v20 - v21 if switches['S2_a'] else i_s2a
        res[9] = v21 - v22 if switches['S2_b'] else i_s2b
        return res

    sol = root(residual, initial_guess, method='hybr', options={'maxfev': 100})
    
    if not sol.success:
        total_current = (flat_G[0] + flat_G[1] + flat_G[2]) / 3000.0 * ISC
        return max(0.0, total_current), initial_guess

    v10_f = sol.x[0]
    total_current = (solve_module_current_with_bypass(V_array - v10_f, flat_G[0], pv_params) +
                     solve_module_current_with_bypass(V_array - sol.x[1], flat_G[1], pv_params) +
                     solve_module_current_with_bypass(V_array - sol.x[2], flat_G[2], pv_params))
    return max(0.0, total_current), list(sol.x)

def generate_dataset(num_samples):
    switch_names = ['S1_a', 'S1_b', 'S2_a', 'S2_b']
    all_combos = list(itertools.product([True, False], repeat=4))
    dataset = []
    np.random.seed(42)
    
    linear_voltages = np.linspace(10.0, 110.0, 25)

    for sample in range(num_samples):
        # Force print to console immediately so you can see it working
        print(f"-> Processing {sample + 1}/{num_samples}...", flush=True)
            
        shading = np.random.choice(np.arange(100, 1100, 100), size=(3, 3))
        flat_shading = shading.flatten()
        
        best_power = -1
        best_config_idx = 0
        
        for idx, combo in enumerate(all_combos):
            switches = dict(zip(switch_names, combo))
            guess_config = [2*linear_voltages[0]/3]*3 + [linear_voltages[0]/3]*3 + [0.0]*4
            
            max_p = 0
            for v in linear_voltages:
                i_out, guess_config = solve_mna_network(v, shading, switches, pv_params, guess_config)
                p_out = v * i_out
                if p_out > max_p: max_p = p_out
            if max_p > best_power:
                best_power = max_p
                best_config_idx = idx
                
        baseline_switches = {'S1_a': True, 'S1_b': True, 'S2_a': True, 'S2_b': True}
        guess = [2*linear_voltages[0]/3]*3 + [linear_voltages[0]/3]*3 + [0.0]*4
        
        temp_powers = []
        temp_rows = []
        
        for v_terminal in linear_voltages:
            i_out, guess = solve_mna_network(v_terminal, shading, baseline_switches, pv_params, guess)
            p_terminal = v_terminal * i_out
            temp_powers.append(p_terminal)
            
            row_data = {
                'Output_Voltage': v_terminal,
                'Output_Power': p_terminal,
                'Optimal_Config_Label': best_config_idx
            }
            for i in range(9):
                row_data[f'Mod_{i}_G'] = flat_shading[i]
            temp_rows.append(row_data)
            
        # Median adjustment
        valid_powers = [p for p in temp_powers if p > 0.01]
        local_median_power = np.median(valid_powers) if len(valid_powers) > 0 else 0.0
        
        for row in temp_rows:
            if row['Output_Power'] <= 0.01:
                row['Output_Power'] = local_median_power
            dataset.append(row)
        
    return pd.DataFrame(dataset)

if __name__ == "__main__":
    print("Initializing engine loop with linear 10V-110V mapping...", flush=True)
    df = generate_dataset(num_samples=150)
    
    df.to_csv("pv_shading_dataset.csv", index=False)
    print(f"\nSuccess! Built dataset with {len(df)} total rows.")

Initializing engine loop with linear 10V-110V mapping...
-> Processing 1/150...
-> Processing 2/150...
-> Processing 3/150...
-> Processing 4/150...
-> Processing 5/150...
-> Processing 6/150...
-> Processing 7/150...
-> Processing 8/150...
-> Processing 9/150...
-> Processing 10/150...
-> Processing 11/150...
-> Processing 12/150...
-> Processing 13/150...
-> Processing 14/150...
-> Processing 15/150...
-> Processing 16/150...
-> Processing 17/150...
-> Processing 18/150...
-> Processing 19/150...
-> Processing 20/150...
-> Processing 21/150...
-> Processing 22/150...
-> Processing 23/150...
-> Processing 24/150...
-> Processing 25/150...
-> Processing 26/150...
-> Processing 27/150...
-> Processing 28/150...
-> Processing 29/150...
-> Processing 30/150...
-> Processing 31/150...
-> Processing 32/150...
-> Processing 33/150...
-> Processing 34/150...
-> Processing 35/150...
-> Processing 36/150...
-> Processing 37/150...
-> Processing 38/150...
-> Processing 39/150...
-> Processing 40